In [25]:
import pandas as pd
import numpy as np
import wandb
from IPython.display import display

In [26]:
ENTITY   = "bombaclat-mpi"
PROJECTS = {
    "persona":    "ma-steering-lora-nothink-v3",
    "nopersona":  "ma-steering-lora-nothink-v3-nopersona",
}
METRIC = "reward/raw_cos_sim_mean"

api = wandb.Api()

def fetch_runs(entity, project, label):
    rows = []
    for run in api.runs(f"{entity}/{project}"):
        hist = run.history(keys=[METRIC], x_axis="_step", pandas=False)
        vals = [(r["_step"], r[METRIC]) for r in hist if r.get(METRIC) is not None]
        if not vals:
            continue
        first_step, first_val = vals[0]
        last_step,  last_val  = vals[-1]
        rows.append({
            "condition":  label,
            "run_name":   run.name,
            "first_step": int(first_step),
            "first_val":  first_val,
            "last_step":  int(last_step),
            "last_val":   last_val,
            "delta":      last_val - first_val,
            "n_steps":    int(last_step),
        })
    return rows

def fetch_redteam(entity, project):
    rows = []
    for run in api.runs(f"{entity}/{project}"):
        s = run.summary
        mean_cos = s.get("summary/mean_target_cos_sim") or s.get("reward/raw_cos_sim_mean")
        if mean_cos is None or run.state == "crashed":
            continue
        rows.append({
            "condition":  "redteam",
            "run_name":   run.name,
            "mean_cos":   mean_cos,
            "state":      run.state,
        })
    return rows

raw = fetch_runs(ENTITY, PROJECTS["persona"],    "persona")
raw += fetch_runs(ENTITY, PROJECTS["nopersona"], "nopersona")
rt_raw = fetch_redteam(ENTITY, "redteam-baseline")

print(f"training: {len(raw)} runs  |  redteam: {len(rt_raw)} runs")

fetched 32 runs


In [ ]:
# ── Diagnostic: inspect what wandb actually returns ─────────────────────
print("=== TRAINING RUNS ===")
for r in sorted(raw, key=lambda x: x["run_name"]):
    print("{:40s}  start={:+.4f}  end={:+.4f}  steps={:4d}  delta={:+.4f}".format(
        r["run_name"], r["first_val"], r["last_val"], r["n_steps"], r["delta"]))

print()
print("=== REDTEAM RUNS ===")
for r in api.runs(f"{ENTITY}/redteam-baseline"):
    s = r.summary
    keys_present = [k for k in s.keys() if "cos" in k.lower() or "target" in k.lower()]
    mean_cos = s.get("summary/mean_target_cos_sim") or s.get("reward/raw_cos_sim_mean")
    print("{:40s}  state={:8s}  mean_cos={:s}  keys={:s}".format(
        r.name, r.state,
        "{:+.4f}".format(mean_cos) if mean_cos is not None else "None",
        str(keys_present[:5])))


In [27]:
import re

def parse_training_name(run_name):
    m = re.match(r"ipdNV3(?:np)?_(max|min)_(.*)", run_name)
    if not m:
        return None, None
    direction = m.group(1)
    trait = m.group(2).replace("_", " ")
    return trait, direction

def parse_redteam_name(run_name):
    # rt_evil_max  /  rt_adaptable-flexible_max
    m = re.match(r"rt_(.+)_(max|min)$", run_name)
    if not m:
        return None, None
    trait = m.group(1).replace("-", " ")
    direction = m.group(2)
    return trait, direction

records = []
for r in raw:
    trait, direction = parse_training_name(r["run_name"])
    if trait is None:
        continue
    records.append({
        "trait":      trait,
        "dir":        direction,
        "condition":  r["condition"],
        "start":      r["first_val"],
        "end":        r["last_val"],
        "steps":      r["n_steps"],
        "delta":      r["delta"],
    })

rt_records = []
for r in rt_raw:
    trait, direction = parse_redteam_name(r["run_name"])
    if trait is None:
        continue
    rt_records.append({
        "trait":     trait,
        "dir":       direction,
        "mean_cos":  r["mean_cos"],
        "state":     r["state"],
    })

flat    = pd.DataFrame(records)
flat_rt = pd.DataFrame(rt_records)
print("redteam traits:", sorted(flat_rt[["trait","dir"]].apply(tuple, axis=1).tolist()))

In [28]:
p   = flat[flat.condition == "persona"].set_index(["trait","dir"])[["start","end","steps","delta"]]
np_ = flat[flat.condition == "nopersona"].set_index(["trait","dir"])[["start","end","steps","delta"]]
rt  = flat_rt.set_index(["trait","dir"])[["mean_cos"]]

p.columns   = pd.MultiIndex.from_tuples([("Persona LoRA",    c) for c in p.columns])
np_.columns = pd.MultiIndex.from_tuples([("No-Persona LoRA", c) for c in np_.columns])
rt.columns  = pd.MultiIndex.from_tuples([("Redteam (Grok)",  c) for c in rt.columns])

df = p.join(np_, how="outer").join(rt, how="outer")
df = df.sort_index(level="trait")

# Grok delta = mean_cos minus the less-favourable (higher) training start.
# For "min" rows we negate so that green = Grok moved in the intended direction
# (mean_cos went below baseline -> raw delta < 0 -> negated -> positive/green).
p_start  = df[("Persona LoRA",    "start")]
np_start = df[("No-Persona LoRA", "start")]
grok_baseline = p_start.combine(np_start,
    lambda a, b: (a if (not np.isnan(a) and (np.isnan(b) or a >= b)) else b))
grok_delta_raw = df[("Redteam (Grok)", "mean_cos")] - grok_baseline
min_mask = df.index.get_level_values("dir") == "min"
df[("Redteam (Grok)", "delta")] = grok_delta_raw.where(~min_mask, -grok_delta_raw)

df.head(3)


<bound method NDFrame.head of                          Persona LoRA                            \
                                start       end steps     delta   
trait                dir                                          
angry                max    -0.146725 -0.073034    97  0.073691   
agreeableness        min    -0.010182  0.059970    88  0.070152   
                     max     0.025827  0.056732   112  0.030905   
ethical              max    -0.077408 -0.053958   178  0.023450   
adaptable            min     0.133288  0.155136   195  0.021848   
evil                 max    -0.070609 -0.052165   260  0.018444   
assertive            min    -0.041026 -0.023766    47  0.017260   
apathetic            max    -0.236814 -0.219910    45  0.016904   
angry                min     0.145240  0.159996   127  0.014756   
adaptable flexible   max    -0.181918 -0.168224   208  0.013694   
assertive            max     0.032427  0.043835    54  0.011408   
adaptable            max    -0.1

In [29]:
def color_delta(val):
    """Training delta: green/red by sign, intensity by magnitude."""
    if pd.isna(val):
        return "color: #bbb"
    if abs(val) < 0.01:
        tint = "hsl(130,30%,94%)" if val >= 0 else "hsl(0,30%,94%)"
        return "background-color: {}; color: #888".format(tint)
    if val >= 0:
        intensity = min(abs(val) / 0.075, 1.0)
        lightness = int(60 - intensity * 28)
        return "background-color: hsl(130,52%,{}%); color: white; font-weight: 600".format(lightness)
    return "background-color: hsl(0,62%,44%); color: white; font-weight: 600"

def color_cos(val):
    """Absolute cos sim: green if positive, red if negative. Scale: 0.25 = max intensity."""
    if pd.isna(val):
        return "color: #bbb"
    if abs(val) < 0.01:
        return "color: #aaa"
    intensity = min(abs(val) / 0.25, 1.0)
    lightness = int(60 - intensity * 28)
    if val >= 0:
        return "background-color: hsl(130,52%,{}%); color: white; font-weight: 600".format(lightness)
    return "background-color: hsl(0,55%,{}%); color: white; font-weight: 600".format(lightness)

def group_borders(data):
    result = pd.DataFrame("", index=data.index, columns=data.columns)
    for i in range(1, len(data)):
        if data.index[i][0] != data.index[i-1][0]:
            result.iloc[i, :] = "border-top: 2px solid #bbb"
    return result

end_cols   = [("Persona LoRA", "end"), ("No-Persona LoRA", "end")]
delta_cols = [("Persona LoRA", "delta"), ("No-Persona LoRA", "delta"), ("Redteam (Grok)", "delta")]
rt_cols    = [("Redteam (Grok)", "mean_cos")]

fmt = {}
for cond in ["Persona LoRA", "No-Persona LoRA"]:
    fmt[(cond, "start")] = "{:+.4f}"
    fmt[(cond, "end")]   = "{:+.4f}"
    fmt[(cond, "steps")] = "{:.0f}"
    fmt[(cond, "delta")] = "{:+.4f}"
fmt[("Redteam (Grok)", "mean_cos")] = "{:+.4f}"
fmt[("Redteam (Grok)", "delta")]    = "{:+.4f}"

styled = (
    df.style
    .format(fmt, na_rep="—")
    .map(color_delta, subset=delta_cols)
    .apply(group_borders, axis=None)
    .set_table_styles([
        {"selector": "th",
         "props": "background: #f7f7f7; font-size: 12px; padding: 6px 10px; "
                  "border-bottom: 1px solid #ddd; text-align: center;"},
        {"selector": "th.col_heading.level0",
         "props": "font-size: 13px; font-weight: bold; border-bottom: 2px solid #aaa;"},
        {"selector": "th.row_heading",
         "props": "text-align: left; font-size: 12px; background: #f7f7f7;"},
        {"selector": "td",
         "props": "font-size: 12px; padding: 5px 10px; font-family: monospace; text-align: right;"},
    ])
    .set_caption(
        "end / mean_cos share the same color scale (±0.25). "
        "Redteam = mean cos sim of untrained model under Grok-4 adversarial pressure (no start available). "
        "Training delta = end − start."
    )
)

display(styled)

In [30]:
# Summary
p_delta_mean  = df[("Persona LoRA",    "delta")].mean()
np_delta_mean = df[("No-Persona LoRA", "delta")].mean()
rt_delta_mean = df[("Redteam (Grok)",  "delta")].mean()
n_persona   = df[("Persona LoRA",    "delta")].notna().sum()
n_nopersona = df[("No-Persona LoRA", "delta")].notna().sum()
n_grok      = df[("Redteam (Grok)",  "delta")].notna().sum()

# Persona vs No-Persona (paired)
paired_pnp = df.dropna(subset=[("Persona LoRA", "delta"), ("No-Persona LoRA", "delta")])
p_wins   = (paired_pnp[("Persona LoRA",    "delta")] > paired_pnp[("No-Persona LoRA", "delta")]).sum()
np_wins  = (paired_pnp[("No-Persona LoRA", "delta")] > paired_pnp[("Persona LoRA",    "delta")]).sum()

# Grok vs Persona (paired)
paired_gp = df.dropna(subset=[("Redteam (Grok)", "delta"), ("Persona LoRA", "delta")])
grok_vs_p_wins = (paired_gp[("Redteam (Grok)", "delta")] > paired_gp[("Persona LoRA",    "delta")]).sum()
p_vs_grok_wins = (paired_gp[("Persona LoRA",    "delta")] > paired_gp[("Redteam (Grok)", "delta")]).sum()

# Grok vs No-Persona (paired)
paired_gnp = df.dropna(subset=[("Redteam (Grok)", "delta"), ("No-Persona LoRA", "delta")])
grok_vs_np_wins = (paired_gnp[("Redteam (Grok)", "delta")] > paired_gnp[("No-Persona LoRA", "delta")]).sum()
np_vs_grok_wins = (paired_gnp[("No-Persona LoRA", "delta")] > paired_gnp[("Redteam (Grok)", "delta")]).sum()

print("Mean delta (signed, higher = moved in intended direction)")
print("  Persona LoRA    : {:+.4f}  (n={})".format(p_delta_mean,  n_persona))
print("  No-Persona LoRA : {:+.4f}  (n={})".format(np_delta_mean, n_nopersona))
print("  Redteam (Grok)  : {:+.4f}  (n={})".format(rt_delta_mean, n_grok))
print()
print("Head-to-head wins (per trait×dir pair)")
print("  Persona vs No-Persona  : {} - {}  (n={})".format(p_wins,         np_wins,        len(paired_pnp)))
print("  Grok    vs Persona     : {} - {}  (n={})".format(grok_vs_p_wins,  p_vs_grok_wins,  len(paired_gp)))
print("  Grok    vs No-Persona  : {} - {}  (n={})".format(grok_vs_np_wins, np_vs_grok_wins, len(paired_gnp)))


Persona    mean delta : +0.0171  (n=18)
No-Persona mean delta : +0.0228  (n=14)

Persona    wins (paired) : 7 / 14
No-Persona wins (paired) : 7 / 14
